In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 5.14 Heat Engines and Thermodynamic Cycles

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume V — Classical Statistical Mechanics",
    number="5.14",
    title="Heat Engines and Thermodynamic Cycles",
    blurb="The subject thermodynamics was invented for: turning heat into "
    "work, and the law that says how much must be wasted. Carnot's bound "
    "built leg by leg on the ideal gas, the Otto cycle inside every "
    "engine, the arithmetic that makes heat pumps look like magic, and "
    "the endoreversible square root that predicts real power-plant "
    "efficiencies better than Carnot ever could.",
    difficulty="intermediate",
    estimate="120–150 min",
)

## Notebook overview

Thermodynamics did not begin with entropy; it began with an engineering
question. Sadi Carnot asked in 1824 {cite}`carnot1824` how much work a
steam engine could possibly extract from a given flow of heat, and his
answer — a bound that depends only on two temperatures, not on the
working substance, the pressure, or the cleverness of the design — is
among the most consequential single results in physics. This volume has
so far run the logic the modern way, from microstates to entropy to free
energies ([§5.4](microstates-entropy-temperature.ipynb),
[§5.7](potentials-legendre-maxwell.ipynb)); this notebook closes the
loop and spends that machinery on the subject it was invented for.

The plan is to *build* the cycles, leg by quadrature-checked leg, on the
ideal gas whose every property [§5.6](ideal-gas-fundamental-relation.ipynb)
certified. Carnot's cycle delivers its bound exactly; the Otto cycle
prices the engine in every car; the same arithmetic run backwards
explains why a heat pump beats a resistance heater several times over.
And because the course prizes honesty about idealization, the finale is
the *endoreversible* engine of Curzon and Ahlborn {cite}`curzon1975`:
let the heat flow through finite conductances — as it must in any
machine that runs at nonzero power — and the efficiency at maximum
power drops from Carnot's $1 - T_c/T_h$ to the startling
$1 - \sqrt{T_c/T_h}$, a formula with no adjustable constants that lands
within a few points of real coal, nuclear, and geothermal plants. The
volume's standing reference remains {cite}`nolting6`.

A note on reading the checks in this notebook: a validation compares a
result to an expected physical fact. A ✗ does not by itself mean the
answer is wrong; it means the output did not match what the check
expected, which may be a genuine error, a different-but-valid
convention, or too tight a tolerance. Treat a ✗ as a prompt to locate
the discrepancy. Passing is strong evidence, not proof.

## Theory in brief

**The working substance.** Throughout, the working gas is the monatomic
ideal gas of [§5.6](ideal-gas-fundamental-relation.ipynb) in units
$Nk_B = 1$: equation of state $pV = T$, internal energy $U = \tfrac32
T$, adiabatic index $\gamma = 5/3$, and entropy (up to a constant)
$S = \tfrac32\ln T + \ln V$. Quasi-static work is $W = \int p\,dV$, and
the first law $\Delta U = Q - W$ prices every leg of every cycle. Two
leg types recur: the **isotherm**, along which $\Delta U = 0$ so
$Q = W = T\ln(V_f/V_i)$, and the **adiabat**, along which $Q = 0$,
$TV^{\gamma-1}$ is constant, and $W = -\Delta U = \tfrac32(T_i - T_f)$.

**The Carnot cycle and the bound.** Two isotherms at $T_h$ and $T_c$
joined by two adiabats. Because heat enters only at $T_h$ and leaves
only at $T_c$, the entropy bookkeeping is a rectangle in the $T$–$S$
plane, and the efficiency $\eta = W_{\rm net}/Q_h$ is

```{math}
:label: eq-he-carnot
\eta_{\rm C} \;=\; 1 - \frac{T_c}{T_h},
```

independent of everything but the two reservoir temperatures. The
second law forbids any engine between the same reservoirs from doing
better; the $T$–$S$ rectangle makes the geometry of the claim visible
(net work = enclosed area, in *either* the $p$–$V$ or the $T$–$S$
plane).

**The Otto cycle.** Two adiabats joined by two isochores (constant
volume) — the idealization of the spark-ignition engine, where the
compression ratio $r = V_{\max}/V_{\min}$ is the design parameter.
Working the four legs gives

```{math}
:label: eq-he-otto
\eta_{\rm Otto} \;=\; 1 - r^{1-\gamma},
```

with $\gamma \approx 1.4$ for the air the real engine breathes.

**Refrigerators and heat pumps.** Run any engine backwards and work
pumps heat uphill. The figure of merit is the coefficient of
performance: for a refrigerator ${\rm COP}_{\rm fr} = Q_c/W$, for a
heat pump ${\rm COP}_{\rm hp} = Q_h/W$, with the reversible (Carnot)
limits

```{math}
:label: eq-he-cop
{\rm COP}_{\rm fr} \;=\; \frac{T_c}{T_h - T_c},
\qquad
{\rm COP}_{\rm hp} \;=\; \frac{T_h}{T_h - T_c}
\;=\; {\rm COP}_{\rm fr} + 1 ,
```

the last identity being the first law wearing a party hat: every joule
of work ends up as delivered heat too.

**Endoreversibility: the Curzon–Ahlborn square root.** Carnot's engine
attains its bound only reversibly — infinitely slowly, at zero power.
A real plant must push heat through finite thermal conductances, so the
working fluid runs *hotter than the cold reservoir and colder than the
hot one* ($T_c < T_{cw} < T_{hw} < T_h$), reversible inside, irreversible
at the contacts. Maximizing the *power* rather than the efficiency over
the intermediate temperatures gives, remarkably independently of the
conductances,

```{math}
:label: eq-he-ca
\eta_{\rm CA} \;=\; 1 - \sqrt{\frac{T_c}{T_h}} ,
```

Curzon and Ahlborn's result {cite}`curzon1975` (anticipated by
Novikov and by Chambadal in the 1950s). It is the rare formula that
*improves* on an exact bound as a description of reality: real plants
are built to maximize power per dollar, not efficiency, and their
observed efficiencies track {eq}`eq-he-ca` far better than
{eq}`eq-he-carnot`.

## Setup

The monatomic ideal gas in $Nk_B = 1$ units throughout; $\gamma = 5/3$
except where the Otto cycle breathes diatomic air ($\gamma = 1.4$),
stated where used. Nothing in this notebook is stochastic.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from scipy.integrate import quad
from scipy.optimize import minimize_scalar

from ecp import animate, validate

GAMMA_MONO = 5.0 / 3.0  # monatomic ideal gas
GAMMA_AIR = 1.4  # diatomic air, for the Otto cycle


def pressure_isotherm(V, T):
    """Ideal-gas pressure p = T/V along an isotherm (N kB = 1 units).

    The equation of state pV = T solved for p at fixed temperature: the
    curve every isothermal leg of every cycle in this notebook follows.

    Parameters
    ----------
    V : float or numpy.ndarray
        Volume(s).
    T : float
        Temperature.

    Returns
    -------
    float or numpy.ndarray
        Pressure(s).
    """
    return T / V


def pressure_adiabat(V, p_ref, V_ref, gamma=GAMMA_MONO):
    """Pressure along an adiabat p V^gamma = const, anchored at (p_ref, V_ref).

    The quasi-static curve of a thermally isolated leg; steeper than the
    isotherm through the same point by the factor gamma.

    Parameters
    ----------
    V : float or numpy.ndarray
        Volume(s).
    p_ref, V_ref : float
        Anchor point the adiabat passes through.
    gamma : float
        Adiabatic index.

    Returns
    -------
    float or numpy.ndarray
        Pressure(s).
    """
    return p_ref * (V_ref / V) ** gamma


def entropy_ideal(T, V):
    """Ideal-gas entropy S = (3/2) ln T + ln V up to a constant (N kB = 1).

    The Sackur–Tetrode expression of §5.6 stripped of its additive
    constant, which cancels in every difference this notebook takes.

    Parameters
    ----------
    T : float or numpy.ndarray
        Temperature(s).
    V : float or numpy.ndarray
        Volume(s).

    Returns
    -------
    float or numpy.ndarray
        Entropy up to an additive constant.
    """
    return 1.5 * np.log(T) + np.log(V)

## Exercise 1 — The Carnot cycle, leg by leg

The bound {eq}`eq-he-carnot` deserves to be *earned*, not quoted: build
the cycle explicitly and let quadrature do the accounting. Take
$T_h = 500$, $T_c = 300$, and the hot isotherm running from $V_1 = 1$
to $V_2 = 2$; the two adiabats then fix $V_3 = V_2 (T_h/T_c)^{3/2}$ and
$V_4 = V_1 (T_h/T_c)^{3/2}$ (from $TV^{\gamma-1}$ constant with
$\gamma = 5/3$).

**Part a)** Compute the work of each leg: the two isothermal legs by
`scipy.integrate.quad` of $p(V) = T/V$ (verify each against the closed
form $T\ln(V_f/V_i)$ to `rtol=1e-10`), the two adiabatic legs from the
first law $W = \tfrac32(T_i - T_f)$. Verify the two adiabatic works
cancel exactly, and that the cycle's net work equals
$(T_h - T_c)\ln(V_2/V_1) = 138.629$ to `rtol=1e-10`.

**Part b)** Verify the bound is attained: $\eta = W_{\rm net}/Q_h$
equals $1 - T_c/T_h = 0.4$ to `rtol=1e-12`, and verify the geometric
reading: the entropy change $\Delta S = \ln(V_2/V_1)$ of the hot
isotherm (from `entropy_ideal`) times $(T_h - T_c)$ — the area of the
$T$–$S$ rectangle — reproduces the same net work (`rtol=1e-12`). Verify
entropy returns exactly to its starting value around the loop
(`atol=1e-14`): the state function came home.

**Part c)** Draw the cycle in both planes: the $p$–$V$ loop (isotherms
and adiabats as curves, the enclosed area shaded) beside the $T$–$S$
rectangle. One irregular loop and one rectangle, the same area: the
$T$–$S$ plane is where Carnot's argument lives.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    W_hot,
    T_H * np.log(V2 / V1),
    "quadrature of p = T/V reproduces the closed-form isothermal work " "T ln(V2/V1)",
    rtol=1e-10,
)
validate.check(
    abs(W_ad_down + W_ad_up) < 1e-12,
    "the two adiabatic works cancel exactly: what the expansion gives, "
    "the compression takes back",
    f"sum = {W_ad_down + W_ad_up:.1e}",
)
validate.close(
    eta_carnot,
    1.0 - T_C / T_H,
    "the constructed cycle attains Carnot's bound 1 - Tc/Th = 0.4 " "exactly",
    rtol=1e-12,
)
validate.close(
    area_TS,
    W_net,
    "and the T–S rectangle's area IS the net work: the geometric form "
    "of the argument",
    rtol=1e-12,
)
validate.check(
    abs(S_loop) < 1e-14,
    "entropy returns exactly around the loop: a state function came " "home",
    f"loop sum {S_loop:.1e}",
)

## Exercise 2 — The Otto cycle: the engine in the driveway

{eq}`eq-he-otto` prices every spark-ignition engine. The idealized
cycle: adiabatic compression from $V_{\max}$ to $V_{\min}$ (the piston
rising), isochoric heat injection at $V_{\min}$ (the spark), adiabatic
expansion back to $V_{\max}$ (the power stroke), isochoric heat
rejection (the exhaust). Take compression ratio $r = 10$, air
($\gamma = 1.4$), intake at $T_1 = 300$, and a spark that heats the
compressed gas by $\Delta T = 1500$ (with $c_V = 1/(\gamma - 1) = 2.5$
in these units).

**Part a)** March through the four corners with the adiabatic relation
$T V^{\gamma-1}$ constant, form $Q_{\rm in} = c_V\,\Delta T$,
$Q_{\rm out} = c_V (T_4 - T_1)$, and verify the leg-by-leg efficiency
$1 - Q_{\rm out}/Q_{\rm in}$ equals the closed form
$1 - r^{1-\gamma} = 0.6019$ to `rtol=1e-12` — and that it does *not*
depend on $\Delta T$: verify the same efficiency (to `rtol=1e-12`) at
$\Delta T = 800$. The compression ratio is the whole game, which is
why diesels (higher $r$) are more efficient.

**Part b)** The honesty paragraph, quantified: real engines deliver
roughly $0.25$–$0.35$, not $0.60$ — friction, finite-time combustion,
heat loss through cylinder walls, and the impossibility of truly
adiabatic strokes. Verify the idealization gap is *large*: the ideal
$0.6019$ exceeds a representative real value $0.30$ by more than $80\%$
of the latter. Idealized cycles are bounds and design compasses, not
predictions; the endoreversible engine of Exercise 4 begins to close
this gap.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    eta_otto_1500,
    eta_otto_exact,
    "the four legs assemble to the closed form 1 - r^(1-γ) = 0.602",
    rtol=1e-12,
)
validate.close(
    eta_otto_800,
    eta_otto_exact,
    "and the spark strength drops out entirely: the compression ratio "
    "is the whole game",
    rtol=1e-12,
)
validate.check(
    eta_otto_exact > 1.8 * REAL_ENGINE,
    "the idealization gap is large: real engines deliver about half the "
    "ideal figure — bounds are not predictions",
    f"{eta_otto_exact:.3f} vs ~{REAL_ENGINE}",
)

## Exercise 3 — Backwards: refrigerators and the heat-pump bargain

Reverse the cycle and work pumps heat from cold to hot;
{eq}`eq-he-cop` prices it. A kitchen refrigerator holds its interior
at $T_c = 275\ \mathrm K$ against a room at $T_h = 295\ \mathrm K$; a
heat pump warms the same room against the same $20\ \mathrm K$ gap.

**Part a)** Verify the reversible figures: ${\rm COP}_{\rm fr} =
275/20 = 13.75$ and ${\rm COP}_{\rm hp} = 295/20 = 14.75$
(`rtol=1e-12` each), and the first-law identity ${\rm COP}_{\rm hp} -
{\rm COP}_{\rm fr} = 1$ *exactly* (`atol=1e-12`): the work you pay
arrives in the room as heat, every joule of it.

**Part b)** The bargain, and the honesty. A heat pump delivering
$Q_h$ uses $Q_h/{\rm COP}$ of work while a resistance heater uses
$Q_h$ outright; even at a *real* seasonal COP of $3.5$ (against the
reversible $14.75$ — the usual factor-of-four honesty gap), verify the
heat pump heats the room with $3.5\times$ less electricity than the
resistor, and that the reversible bound would allow $14.75\times$.
Verify also the trend that matters for cold climates: recomputing the
reversible ${\rm COP}_{\rm hp}$ at an outdoor temperature of
$255\ \mathrm K$ ($-18\ \mathrm{^\circ C}$) drops it from $14.75$ to
$7.375$ — exactly a factor $2$ (`rtol=1e-12`), because the gap
doubled. Pumping uphill costs in proportion to the hill.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([cop_fr, cop_hp]),
    np.array([13.75, 14.75]),
    "the reversible coefficients: 13.75 to keep the milk cold, 14.75 "
    "to warm the room",
    rtol=1e-12,
)
validate.check(
    abs((cop_hp - cop_fr) - 1.0) < 1e-12,
    "COP_hp - COP_fr = 1 exactly: the paid work arrives as heat too",
    f"difference {cop_hp - cop_fr:.12f}",
)
validate.check(
    REAL_COP > 1.0 and cop_hp / REAL_COP > 4.0,
    "even a real heat pump beats resistance heating 3.5-fold, while "
    "sitting a factor ~4 below its reversible bound",
    f"real {REAL_COP} vs reversible {cop_hp:.2f}",
)
validate.close(
    cop_hp / cop_hp_winter,
    2.0,
    "and doubling the temperature gap exactly halves the reversible "
    "COP: pumping uphill costs in proportion to the hill",
    rtol=1e-12,
)

## Exercise 4 — The Curzon–Ahlborn engine: power against perfection

Now the finale's physics. Model the endoreversible engine the way
Curzon and Ahlborn did {cite}`curzon1975`: a Carnot engine runs
reversibly between *internal* temperatures $T_{hw}$ and $T_{cw}$,
while the heat it uses must leak in and out through finite
conductances from the true reservoirs at $T_h$ and $T_c$. In the
steady state the reversible core forces the entropy balance
$Q_h/T_{hw} = Q_c/T_{cw}$, and with Newton's-law heat flow
($Q \propto$ temperature drop) the power $P = Q_h - Q_c$ becomes a
function of a single free internal temperature.

**Part a)** Implement the model with equal conductances: for a trial
$T_{hw} \in (T_c, T_h)$, take $Q_h = T_h - T_{hw}$, solve the entropy
balance for $T_{cw}$, form $P = Q_h - Q_c$, and maximize with
`scipy.optimize.minimize_scalar` (bounded) at $T_h = 500$,
$T_c = 300$. Verify the efficiency *at maximum power*,
$1 - T_{cw}/T_{hw}$, equals the closed form
$1 - \sqrt{T_c/T_h} = 0.22540$ to `rtol=1e-5` — the square root
emerging from an optimization that never mentions it — and verify it
sits well below Carnot's $0.4$: the price of running at nonzero power.

**Part b)** The confrontation with reality, using Curzon and
Ahlborn's own comparison set: the West Thurrock coal station
($T_h = 838$, $T_c = 298\ \mathrm K$, observed $\eta = 0.36$), the
CANDU nuclear reactor ($573$, $298$, observed $0.30$), and the
Larderello geothermal field ($523$, $353$, observed $0.16$). Verify
that for all three, $|\eta_{\rm CA} - \eta_{\rm obs}| < 0.05$ while
Carnot overshoots by more than $0.15$ for the two steam plants: a
zero-parameter formula out-predicting an exact bound, because it asks
the question the engineers asked.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    eta_maxP,
    eta_ca_exact,
    "maximizing POWER over the internal temperatures lands on "
    "1 - √(Tc/Th): the square root emerges from an optimization that "
    "never mentions it",
    rtol=1e-5,
)
validate.check(
    eta_maxP < (1.0 - T_C / T_H) - 0.1,
    "and it sits far below Carnot: the price of running at nonzero " "power",
    f"{eta_maxP:.3f} vs Carnot {1 - T_C / T_H:.3f}",
)
validate.check(
    all(d < 0.05 for d in ca_devs),
    "Curzon–Ahlborn lands within 0.05 of all three real plants — coal, "
    "nuclear, geothermal — with zero adjustable parameters",
    f"deviations {[round(d, 3) for d in ca_devs]}",
)
validate.check(
    carnot_devs[0] > 0.15 and carnot_devs[1] > 0.15,
    "while Carnot overshoots the two steam plants by more than 15 "
    "points: the exact bound answers a question nobody built for",
    f"Carnot excesses {[round(d, 3) for d in carnot_devs]}",
)

## Exercise 5 — The Stirling cycle and the regenerator's trick

One more cycle earns its place, both for its geometry and for its
moral. The Stirling cycle joins two isotherms by two *isochores*: heat
the gas at constant $V_1$, expand isothermally at $T_h$, cool at
constant $V_2$, compress isothermally at $T_c$. Run naively, the
isochoric legs draw heat at temperatures *below* $T_h$, so the
efficiency falls short of Carnot. The regenerator — Robert Stirling's
1816 trick, decades before thermodynamics existed — stores the
isochoric-cooling heat in a thermal labyrinth and returns *exactly the
same amount* during isochoric heating (the two legs span the same
temperature interval, so the exchange balances identically). With the
isochoric heats internalized, only the isotherms touch the reservoirs,
and Carnot efficiency returns.

**Part a)** With $T_h = 500$, $T_c = 300$, $V_1 = 1$, $V_2 = 2$
(monatomic, $c_V = 3/2$): price the four legs, verify the net work is
$(T_h - T_c)\ln 2 = 138.63$ — identical to Exercise 1's Carnot cycle,
same reservoirs and same $\Delta S$ — and verify the naive efficiency
(isochoric heat counted as bought) is
$W/(Q_{\rm isotherm} + Q_{\rm isochore}) = 0.2144$ (`rtol=1e-4`),
well below Carnot's $0.4$.

**Part b)** Verify the regenerator's exchange balances exactly — the
isochoric heats are equal and opposite (`atol=1e-12`) — and that with
them internalized the efficiency is *exactly* Carnot's $1 - T_c/T_h$
(`rtol=1e-12`). A cycle whose legs are humbler than Carnot's reaches
the same bound by clever bookkeeping: the second law cares only about
where entropy crosses the boundary.

**Part c)** Animate the working loop: the state point traversing the
$p$–$V$ Stirling cycle while a bar gauge tracks the running net work,
resetting each cycle. The animation's physics is certified by Part
a)'s leg accounting, computed from the same path the point traces.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    W_st_net,
    W_net,
    "same reservoirs, same ΔS, same net work as the Carnot cycle of " "Exercise 1",
    rtol=1e-12,
)
validate.close(
    eta_naive,
    0.2144,
    "bought naively, the isochoric heat drags the efficiency to 0.214",
    rtol=1e-4,
)
validate.check(
    abs(Q_isochore_up + Q_isochore_dn) < 1e-12,
    "but the two isochoric heats balance exactly: the regenerator's "
    "exchange is an identity, not an approximation",
    f"sum {Q_isochore_up + Q_isochore_dn:.1e}",
)
validate.close(
    eta_regen,
    1.0 - T_C / T_H,
    "and with them internalized the Stirling cycle attains Carnot "
    "exactly: the second law only counts entropy crossing the boundary",
    rtol=1e-12,
)

In [ ]:
# (solution hidden on the public site)


## Notebook summary

- The Carnot cycle, built leg by leg on the certified ideal gas,
  attained $\eta = 1 - T_c/T_h = 0.4$ to twelve digits, its adiabatic
  works cancelling exactly and its net work $138.63$ equal to the
  $T$–$S$ rectangle's area; entropy closed the loop to $10^{-14}$.
- The Otto cycle's four legs assembled to $1 - r^{1-\gamma} = 0.602$
  at $r = 10$, independent of spark strength to twelve digits — and
  double what real engines deliver, which is the honest size of the
  idealization gap.
- Run backwards, the reversible arithmetic gave ${\rm COP}_{\rm fr} =
  13.75$ and ${\rm COP}_{\rm hp} = 14.75$ with their difference exactly
  $1$; even a real heat pump's $3.5$ beats resistance heating
  $3.5$-fold, and halving the outdoor temperature gap doubles the
  bound.
- Maximizing *power* over the endoreversible engine's internal
  temperatures produced $1 - \sqrt{T_c/T_h}$ numerically to $10^{-5}$
  with no square root put in by hand — and Curzon–Ahlborn's formula
  landed within $0.05$ of the observed efficiencies of a coal, a
  nuclear, and a geothermal plant, where Carnot overshoots by $15$–$28$
  points.
- The Stirling cycle matched Carnot's net work with humbler legs,
  paid $0.214$ efficiency when its isochoric heat was bought — and
  recovered Carnot's $0.4$ exactly once the regenerator's identically
  balanced exchange was internalized.

## Outlook

- **Entropy accounting at full generality.** The Clausius inequality
  $\oint \delta Q/T \le 0$ turns this notebook's equalities into the
  second law's grand bookkeeping; every cycle here saturates it, and
  every real device undershoots.
- **Finite-time thermodynamics.** Curzon–Ahlborn opened a field:
  optimizing cycles under time, conductance, and friction constraints,
  down to the stochastic thermodynamics of single-molecule engines,
  where work and heat become random variables and the second law a
  statement about their distributions — Volume V's fluctuation ideas
  ([§5.9](grand-canonical-ensemble-equivalence.ipynb)) pushed into
  machines.
- **Quantum engines.** Run the cycle on a quantum working substance —
  a spin, a harmonic oscillator ([§7.5](../07-quantum-statistical-mechanics/oscillator-at-temperature.ipynb))
  — and the same bounds reappear with level populations in place of
  volumes: a live research field with tabletop realizations.
- **The arrow home.** The engines here are the practical face of
  [§5.11](taste-of-nonequilibrium.ipynb)'s irreversibility: a cycle
  extracts work precisely because heat flows downhill, and the
  H-theorem's one-way street is the reason the fuel bill never runs
  backwards.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()